# Sensitivity Analysis of models created by model.ipynb

In this notebook, first the model created by model.ipynb is imported and then the sensitivity analysis is implemented.

## importing requried packages

In [1]:
import linopy
import pypsa

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

import cartopy
import cartopy.crs as ccrs

import networkx as nx

import atlite
from atlite.gis import ExclusionContainer, shape_availability
from rasterio.plot import show
from rasterio.crs import CRS
import rasterio as rio

from pathlib import Path
import xarray as xr

## Importing the models

In [2]:
#n = pypsa.Network("pypsa_model_n.nc") # the model without CO2 limitations
n = pypsa.Network("pypsa_model_n_no_co2.nc") # the model with CO2 limitation of 0

INFO:pypsa.network.io:New version 1.0.7 available! (Current: 1.0.5)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, links, loads, storage_units, sub_networks


In [16]:
#n.generators.loc[n.generators.carrier == "solar", "capital_cost"]
n.storage_units.loc[n.storage_units.carrier == "battery storage", "capital_cost"]

name
battery_2h_Alborz      6154.485387
battery_4h_Alborz      9342.928925
battery_6h_Alborz     12531.372464
battery_2h_Ardabil     6154.485387
battery_4h_Ardabil     9342.928925
                          ...     
battery_4h_Yazd        9342.928925
battery_6h_Yazd       12531.372464
battery_2h_Zanjan      6154.485387
battery_4h_Zanjan      9342.928925
battery_6h_Zanjan     12531.372464
Name: capital_cost, Length: 93, dtype: float64

In [3]:
n.generators_t

{'p_min_pu': Empty DataFrame
 Columns: []
 Index: [2025-01-01 00:00:00, 2025-01-01 03:00:00, 2025-01-01 06:00:00, 2025-01-01 09:00:00, 2025-01-01 12:00:00, 2025-01-01 15:00:00, 2025-01-01 18:00:00, 2025-01-01 21:00:00, 2025-01-02 00:00:00, 2025-01-02 03:00:00, 2025-01-02 06:00:00, 2025-01-02 09:00:00, 2025-01-02 12:00:00, 2025-01-02 15:00:00, 2025-01-02 18:00:00, 2025-01-02 21:00:00, 2025-01-03 00:00:00, 2025-01-03 03:00:00, 2025-01-03 06:00:00, 2025-01-03 09:00:00, 2025-01-03 12:00:00, 2025-01-03 15:00:00, 2025-01-03 18:00:00, 2025-01-03 21:00:00, 2025-01-04 00:00:00, 2025-01-04 03:00:00, 2025-01-04 06:00:00, 2025-01-04 09:00:00, 2025-01-04 12:00:00, 2025-01-04 15:00:00, 2025-01-04 18:00:00, 2025-01-04 21:00:00, 2025-01-05 00:00:00, 2025-01-05 03:00:00, 2025-01-05 06:00:00, 2025-01-05 09:00:00, 2025-01-05 12:00:00, 2025-01-05 15:00:00, 2025-01-05 18:00:00, 2025-01-05 21:00:00, 2025-01-06 00:00:00, 2025-01-06 03:00:00, 2025-01-06 06:00:00, 2025-01-06 09:00:00, 2025-01-06 12:00:00, 2025

In [ ]:
# =============================================================================
# Sensitivity Analysis: Technology Cost Variation
# =============================================================================

# 1. Define cost reduction factors
# 1.0 = Original Price, 0.75 = 25% reduction, etc.
cost_factors = [1.0, 0.75, 0.5, 0.25, 0.0]

# 2. Select the technology to analyze
tech_name = "battery storage"  # Change to "solar", "onwind", etc. as needed

# a dictionary to store results
results = {}

# 3. Backup original capital costs to avoid permanent modification during the loop
# Check if the technology is a generator or a storage unit
if tech_name in n.storage_units.carrier.values:
    original_cost = n.storage_units.loc[n.storage_units.carrier == tech_name, "capital_cost"].copy()
    is_storage = True
    print(f"Starting sensitivity analysis for Storage Technology: {tech_name}")
else:
    original_cost = n.generators.loc[n.generators.carrier == tech_name, "capital_cost"].copy()
    is_storage = False
    print(f"Starting sensitivity analysis for Generation Technology: {tech_name}")

# 4. Iterate through cost factors
for factor in cost_factors:
    print(f"Processing: Cost Factor = {factor} ...")
    
    # Update Capital Costs 
    if is_storage:
        n.storage_units.loc[n.storage_units.carrier == tech_name, "capital_cost"] = original_cost * factor
    else:
        n.generators.loc[n.generators.carrier == tech_name, "capital_cost"] = original_cost * factor
     
    # solve the model with co2 limit
    n.optimize(
        #snapshots=n_no_co2.snapshots[:168],  # 1 week
        solver_name="gurobi",
        method=2,
        crossover=0,
        BarConvTol=1.0e-05,
        AggFill=0,
        PreDual=0,
        GURO_PAR_BARDENSETHRESH=200,
        log_to_console=False
    )
    
    # Record Results 
    # Aggregate optimal capacities (p_nom_opt) by carrier
    gen_cap = n.generators.groupby("carrier").p_nom_opt.sum()
    store_cap = n.storage_units.groupby("carrier").p_nom_opt.sum()
    
    # Combine generation and storage capacities
    total_cap = pd.concat([gen_cap, store_cap])
    results[factor] = total_cap

print("Analysis complete. Plotting results...")

# 5. Visualization of Results
df_results = pd.DataFrame(results)

# Plotting a stacked bar chart
# X-axis: Cost Factor, Y-axis: Installed Capacity
ax = df_results.T.plot(kind="bar", stacked=True, figsize=(10, 6))#, colormap="tab20")
print(n.storage_units.carrier.unique())
plt.title(f"Sensitivity Analysis: Impact of {tech_name} Cost Reduction")
plt.xlabel("Cost Factor (1.0 = 100% of Original Price)")
plt.ylabel("Total Installed Capacity [MW]")
plt.legend(title="Technology", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

# Show the plot
plt.show()

Starting sensitivity analysis for Storage Technology: battery storage
Processing: Cost Factor = 1.0 ...


INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.model:Solver options:
 - method: 2
 - crossover: 0
 - BarConvTol: 1e-05
 - AggFill: 0
 - PreDual: 0
 - GURO_PAR_BARDENSETHRESH: 200
 - log_to_console: False
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 8/8 [00:00<00:00, 48.98it/s]
INFO:linopy.io: Writing time: 3.96s


Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2750042


...

INFO:gurobipy:Set parameter PreDual to value 0


Set parameter GURO_PAR_BARDENSETHRESH to value 200


INFO:gurobipy:Set parameter GURO_PAR_BARDENSETHRESH to value 200


Set parameter LogToConsole to value 0


INFO:gurobipy:Set parameter LogToConsole to value 0
INFO:gurobipy:Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[arm] - Darwin 25.2.0 25C56)
INFO:gurobipy:
INFO:gurobipy:CPU model: Apple M4 Pro
INFO:gurobipy:Thread count: 14 physical cores, 14 logical processors, using up to 14 threads
INFO:gurobipy:
INFO:gurobipy:Non-default parameters:
INFO:gurobipy:Method  2
INFO:gurobipy:BarConvTol  1e-05
INFO:gurobipy:Crossover  0
INFO:gurobipy:AggFill  0
INFO:gurobipy:PreDual  0
INFO:gurobipy:LogToConsole  0
INFO:gurobipy:GURO_PAR_BARDENSETHRESH  200
INFO:gurobipy:
INFO:gurobipy:Optimize a model with 5019879 rows, 2193249 columns and 10762913 nonzeros (Min)
INFO:gurobipy:Model fingerprint: 0x57755f16
INFO:gurobipy:Model has 318609 linear objective coefficients
INFO:gurobipy:Coefficient statistics:
INFO:gurobipy:  Matrix range     [2e-10, 7e+02]
INFO:gurobipy:  Objective range  [1e-02, 2e+05]
INFO:gurobipy:  Bounds range     [0e+00, 0e+00]
INFO:gurobipy:  RHS range        [2e+00, 4e+05]
I